In [ ]:
!pip install unsloth trl peft accelerate bitsandbytes datasets mistral_common

In [ ]:
from google.colab import files
uploaded = files.upload()

# Step 3: Load and format dataset
import json
from datasets import Dataset

# Replace with the actual uploaded file name if different
dataset_path = "cq_warehouse_dataset.json"

# Load JSON data
with open(dataset_path, "r") as f:
    data = json.load(f)

# Format prompt for instruction tuning (user prompt → Python code output)
def format_prompt(example):
    return (
        f"<|begin_of_text|>\n"
        f"### Instruction:\n{example['prompt']}\n\n"
        f"### Response:\n{example['completion']}\n"
        f"<|end_of_text|>"
    )

# Apply formatting
formatted_data = [format_prompt(item) for item in data]

# Convert to Hugging Face Dataset
dataset = Dataset.from_dict({"text": formatted_data})

# Optional: Split for training/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Step 4: Inspect a few samples
print(dataset["train"][0]["text"])

In [ ]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detection

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"], # Select the 'train' split
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none", # Disable Weights & Biases logging
    ),
)

In [ ]:
# Train the model
trainer_stats = trainer.train()

In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt (CAD-style example)
messages = [
    {
        "role": "user",
        "content": "Create a countersunk screw of size M5-0.8 length 12mm type iso14582",
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

In [ ]:
!rm -rf llama.cpp
!git clone --recursive https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release
%cd ..

In [ ]:
!mv /content/llama.cpp/build/bin/llama-quantize /content/llama.cpp/

In [ ]:
# Save the fine-tuned model in a standard Hugging Face format
model.save_pretrained("hf_model")
tokenizer.save_pretrained("hf_model")

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model_name = "unsloth/Phi-3-mini-4k-instruct"  # ✅ full-precision version
adapter_path = "/content/hf_model"
merged_path = "/content/merged_model"

# Load base model (full precision)
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,  # ✅ ensure FP16
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Load the LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

# Merge LoRA weights
model = model.merge_and_unload()

# Ensure all tensors are FP16
model = model.to(torch.float16)

# Save merged model
model.save_pretrained(merged_path, safe_serialization=True)
tokenizer.save_pretrained(merged_path)

print("✅ Merged model saved successfully at:", merged_path)


In [ ]:
!python3 /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model --outfile /content/ask-cad.gguf

In [ ]:
!/content/llama.cpp/llama-quantize /content/ask-cad.gguf /content/unsloth.Q4_K_M.gguf q4_k_m

In [ ]:
from google.colab import files
files.download("/content/unsloth.Q4_K_M.gguf")